# Gercek dunya seti: AI uretimi (qwen2.5-coder 7B + llama3.2 3B)

Ucretsiz Kaggle GPU'sunda `scripts/07_real_world.py ai` calistirir. Gereken tek girdi:
`human.jsonl` (430 insan fonksiyonu) - sag panelden veri seti olarak eklenir.
Cikti: `/kaggle/working/ai.jsonl` ve `ai_failed.jsonl` (Output sekmesinden indirilir).

In [ ]:
# 1) Ollama kurulumu ve modeller
!apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
import os, subprocess, time
os.environ["OLLAMA_KEEP_ALIVE"] = "30m"
subprocess.Popen(["ollama", "serve"], stdout=open("/kaggle/working/ollama.log", "w"),
                 stderr=subprocess.STDOUT)
time.sleep(15)
!ollama --version
!ollama pull qwen2.5-coder:7b-instruct
!ollama pull llama3.2:3b
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2) Kod (GitHub) ve insan verisi
import glob, shutil
!rm -rf /tmp/tez && git clone --depth 1 https://github.com/MetinSerinkaya27/tez-ai-kod-tespiti.git /tmp/tez
found = glob.glob("/kaggle/input/**/human.jsonl", recursive=True)
assert found, "human.jsonl bulunamadi: sag panelden 'Add Input' ile yukleyin"
os.makedirs("/tmp/tez/data/real_world", exist_ok=True)
shutil.copy(found[0], "/tmp/tez/data/real_world/human.jsonl")
# Onceki bir calismanin ciktisi varsa kaldigi yerden devam etmek icin geri kopyala
for name in ("ai.jsonl", "ai_failed.jsonl"):
    prev = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    if prev:
        shutil.copy(prev[0], f"/tmp/tez/data/real_world/{name}")
n = sum(1 for _ in open("/tmp/tez/data/real_world/human.jsonl", encoding="utf-8"))
print("insan kaydi:", n)
assert n == 430, "beklenen 430 kayit"
!pip install -q uv

In [ ]:
# 3) Uretim (tahmini 3-5 saat). Her ornek bitince diske yazilir.
%cd /tmp/tez
!PYTHONPATH=src uv run --no-project --python 3.12 --with requests --with tqdm python -u scripts/07_real_world.py ai
!cp data/real_world/ai.jsonl data/real_world/ai_failed.jsonl /kaggle/working/
!wc -l /kaggle/working/ai.jsonl /kaggle/working/ai_failed.jsonl